In [ ]:
from IPython.utils import io
import tqdm.notebook
import os, sys, random
total = 100
with tqdm.notebook.tqdm(total=total) as pbar:
    with io.capture_output() as captured:
      # Instalar rdkit
      !pip -q install rdkit.pypi==2021.9.4
      pbar.update(20)
      # Instalar Pillow
      !pip -q install Pillow
      pbar.update(40)
      # Instalar molplotly
      !pip install molplotly
      pbar.update(60)
      # Instalar jupyter-dash
      !pip install jupyter-dash
      pbar.update(80)
      # Instalar el diseño de aplicación dash
      !pip install dash-bootstrap-components
      pbar.update(100)


  0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import dash_bootstrap_components as dbc
from sys import argv

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import rdMolDescriptors

# Queremos extraer todos los descriptores moleculares de los compuestos identificados por Machine-Learning y filtrados por Lead-Likeness

### Cargar los datos

In [ ]:
url_DATA_G = "/content/drive/MyDrive/Doctorado_Santiago/Set de Datos 2/Objetivo 4/4_Integracion y Benn/Leadlikeness_compounds.csv"
LeadLikeness = pd.read_csv(url_DATA_G, sep=",", encoding='latin1')

In [ ]:
LeadLikeness

,Name,SMILES,Total Molweight,cLogP,H-Acceptors,H-Donors,Total Surface Area,Relative PSA,Rotatable Bonds,Senotherapeutics:,Leadlikeness
0,"(5-(2,4-bis((3S)-3-methylmorpholin-4-yl)pyrido...",CC1COCCN1C2=NC(=NC3=C2C=CC(=N3)C4=CC(=C(C=C4)O...,465.552,2.4865,9,1,351.95,0.23614,5,1,True
1,(E)-4-((2-N-(4-methoxybenzenesulfonyl)amino)st...,COC1=CC=C(C=C1)S(=O)(=O)N=C2C=CC=CC2=CC=C3C=CN...,382.439,1.7590,6,1,292.14,0.22845,3,1,True
2,10-decarbamoylmitomycin C,CC1=C(C(=O)C2=C(C1=O)N3CC4C(C3(C2CO)OC)N4)N,291.306,-1.6656,7,3,197.58,0.45359,2,1,True
3,10-hydroxycamptothecin,CCC1(C2=C(COC1=O)C(=O)N3CC4=C(C3=C2)N=C5C=CC(=...,364.356,0.8381,7,2,246.20,0.31194,1,1,True
4,103D5R,CC(C1=C(C2=C(C=C1)OC(C=C2)(C)C)OC)N3C=NC4=C3C=...,335.406,4.0763,5,0,259.29,0.18824,3,1,True
...,...,...,...,...,...,...,...,...,...,...,...
265,vistusertib,CC1COCCN1C2=NC(=NC3=C2C=CC(=N3)C4=CC(=CC=C4)C(...,462.552,2.5926,9,1,351.05,0.24073,4,1,True
266,withaferin A,CC1=C(C(=O)OC(C1)C(C)C2CCC3C2(CCC4C3CC5C6(C4(C...,470.604,2.4938,6,2,334.36,0.23905,3,1,True
267,withanone,CC1=C(C(=O)OC(C1)C(C)C2(CCC3C2(CCC4C3C5C(O5)C6...,470.604,2.5971,6,2,330.77,0.24165,2,1,True
268,zomepirac glucuronide,CC1=C(N(C(=C1)CC(=O)OC2C(C(C(C(O2)C(=O)O)O)O)O...,467.857,-0.0007,10,4,323.57,0.36589,7,1,True


In [ ]:
# Cargar la base de datos obtenida en CTD
ctd_data = pd.read_csv("/content/drive/MyDrive/Doctorado_Santiago/Set de Datos 2/Objetivo 4/0_DATA/CTD_curado.csv")

In [ ]:
ctd_data

,GeneSymbol,SMILES,ChemicalName,Interaction,InteractionActions,Total Molweight,Molweight,Monoisotopic Mass,cLogP,cLogS,...,Saturated Hetero-Rings,Non-Aromatic Hetero-Rings,Hetero-Aromatic Rings,Amides,Amines,Alkyl-Amines,Aromatic Amines,Aromatic Nitrogens,Basic Nitrogens,Acidic Oxygens
0,AKT1,[B-]12(OC3C(C(OC3(O1)CO)CO)O)OC4C(C(OC4(O2)CO)...,calcium fructoborate,calcium fructoborate affects the expression of...,affects^expression,774.258,367.0900,367.104785,-5.2466,2.630,...,4,4,0,0,0,0,0,0,0,0
1,PTEN,[B-]12(OC3C(C(OC3(O1)CO)CO)O)OC4C(C(OC4(O2)CO)...,calcium fructoborate,calcium fructoborate results in increased expr...,increases^expression,774.258,367.0900,367.104785,-5.2466,2.630,...,4,4,0,0,0,0,0,0,0,0
2,EIF4EBP1,[B-]12(OC3C(C(OC3(O1)CO)CO)O)OC4C(C(OC4(O2)CO)...,calcium fructoborate,calcium fructoborate affects the expression of...,affects^expression,774.258,367.0900,367.104785,-5.2466,2.630,...,4,4,0,0,0,0,0,0,0,0
3,TSC2,[B-]12(OC3C(C(OC3(O1)CO)CO)O)OC4C(C(OC4(O2)CO)...,calcium fructoborate,calcium fructoborate results in increased expr...,increases^expression,774.258,367.0900,367.104785,-5.2466,2.630,...,4,4,0,0,0,0,0,0,0,0
4,TP53,[Be+2].[O-]S(=O)(=O)[O-],beryllium sulfate,beryllium sulfate promotes the reaction [TP53 ...,affects^binding|increases^reaction,105.074,98.0779,97.967380,-3.4103,1.524,...,0,0,0,0,0,0,0,0,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54396,RELA,S1[As]2S[As]3[As]1S[As]2S3,tetraarsenic tetrasulfide,tetraarsenic tetrasulfide results in decreased...,decreases^expression,427.952,427.9520,427.574656,0.0000,-0.530,...,6,6,0,0,0,0,0,0,0,0
54397,RELA,S1[As]2S[As]3[As]1S[As]2S3,tetraarsenic tetrasulfide,tetraarsenic tetrasulfide results in decreased...,decreases^expression,427.952,427.9520,427.574656,0.0000,-0.530,...,6,6,0,0,0,0,0,0,0,0
54398,MTOR,S1[As]2S[As]3[As]1S[As]2S3,tetraarsenic tetrasulfide,tetraarsenic tetrasulfide promotes the reactio...,decreases^expression|increases^reaction,427.952,427.9520,427.574656,0.0000,-0.530,...,6,6,0,0,0,0,0,0,0,0
54399,EIF4EBP1,S1[As]2S[As]3[As]1S[As]2S3,tetraarsenic tetrasulfide,tetraarsenic tetrasulfide promotes the reactio...,decreases^expression|increases^reaction,427.952,427.9520,427.574656,0.0000,-0.530,...,6,6,0,0,0,0,0,0,0,0


In [ ]:
print(ctd_data.columns)

Index(['GeneSymbol', 'SMILES', 'ChemicalName', 'Interaction',
       'InteractionActions', 'Total Molweight', 'Molweight',
       'Monoisotopic Mass', 'cLogP', 'cLogS', 'H-Acceptors', 'H-Donors',
       'Total Surface Area', 'Relative PSA', 'Polar Surface Area',
       'Druglikeness', 'Mutagenic', 'Tumorigenic', 'Reproductive Effective',
       'Irritant', 'Shape Index', 'Molecular Flexibility',
       'Molecular Complexity', 'Fragments', 'Non-H Atoms', 'Non-C/H Atoms',
       'Metal-Atoms', 'Electronegative Atoms', 'Stereo Centers',
       'Rotatable Bonds', 'Rings Closures', 'Aromatic Atoms', 'sp3-Atoms',
       'Small Rings', 'Carbo-Rings', 'Hetero-Rings', 'Saturated Rings',
       'Non-Aromatic Rings', 'Aromatic Rings', 'Saturated Carbo-Rings',
       'Non-Aromatic Carbo-Rings', 'Carbo-Aromatic Rings',
       'Saturated Hetero-Rings', 'Non-Aromatic Hetero-Rings',
       'Hetero-Aromatic Rings', 'Amides', 'Amines', 'Alkyl-Amines',
       'Aromatic Amines', 'Aromatic Nitrogens', 'Bas

In [ ]:
# Seleccionar columnas
KNN_df = ctd_data[['GeneSymbol', 'SMILES', 'ChemicalName', 'Interaction',
       'InteractionActions', 'Total Molweight', 'Molweight',
       'Monoisotopic Mass', 'cLogP', 'cLogS', 'H-Acceptors', 'H-Donors',
       'Total Surface Area', 'Relative PSA', 'Polar Surface Area',
       'Druglikeness', 'Mutagenic', 'Tumorigenic', 'Reproductive Effective',
       'Irritant', 'Shape Index', 'Molecular Flexibility',
       'Molecular Complexity', 'Fragments', 'Non-H Atoms', 'Non-C/H Atoms',
       'Metal-Atoms', 'Electronegative Atoms', 'Stereo Centers',
       'Rotatable Bonds', 'Rings Closures', 'Aromatic Atoms', 'sp3-Atoms',
       'Small Rings', 'Carbo-Rings', 'Hetero-Rings', 'Saturated Rings',
       'Non-Aromatic Rings', 'Aromatic Rings', 'Saturated Carbo-Rings',
       'Non-Aromatic Carbo-Rings', 'Carbo-Aromatic Rings',
       'Saturated Hetero-Rings', 'Non-Aromatic Hetero-Rings',
       'Hetero-Aromatic Rings', 'Amides', 'Amines', 'Alkyl-Amines',
       'Aromatic Amines', 'Aromatic Nitrogens', 'Basic Nitrogens',
       'Acidic Oxygens']]

# Cambiar nombre a columnas
ctd_data.columns = ['GeneSymbol', 'SMILES', 'Name', 'Interaction',
       'InteractionActions', 'Total Molweight', 'Molweight',
       'Monoisotopic Mass', 'cLogP', 'cLogS', 'H-Acceptors', 'H-Donors',
       'Total Surface Area', 'Relative PSA', 'Polar Surface Area',
       'Druglikeness', 'Mutagenic', 'Tumorigenic', 'Reproductive Effective',
       'Irritant', 'Shape Index', 'Molecular Flexibility',
       'Molecular Complexity', 'Fragments', 'Non-H Atoms', 'Non-C/H Atoms',
       'Metal-Atoms', 'Electronegative Atoms', 'Stereo Centers',
       'Rotatable Bonds', 'Rings Closures', 'Aromatic Atoms', 'sp3-Atoms',
       'Small Rings', 'Carbo-Rings', 'Hetero-Rings', 'Saturated Rings',
       'Non-Aromatic Rings', 'Aromatic Rings', 'Saturated Carbo-Rings',
       'Non-Aromatic Carbo-Rings', 'Carbo-Aromatic Rings',
       'Saturated Hetero-Rings', 'Non-Aromatic Hetero-Rings',
       'Hetero-Aromatic Rings', 'Amides', 'Amines', 'Alkyl-Amines',
       'Aromatic Amines', 'Aromatic Nitrogens', 'Basic Nitrogens',
       'Acidic Oxygens']

In [ ]:
ctd_data

,GeneSymbol,SMILES,Name,Interaction,InteractionActions,Total Molweight,Molweight,Monoisotopic Mass,cLogP,cLogS,...,Saturated Hetero-Rings,Non-Aromatic Hetero-Rings,Hetero-Aromatic Rings,Amides,Amines,Alkyl-Amines,Aromatic Amines,Aromatic Nitrogens,Basic Nitrogens,Acidic Oxygens
0,AKT1,[B-]12(OC3C(C(OC3(O1)CO)CO)O)OC4C(C(OC4(O2)CO)...,calcium fructoborate,calcium fructoborate affects the expression of...,affects^expression,774.258,367.0900,367.104785,-5.2466,2.630,...,4,4,0,0,0,0,0,0,0,0
1,PTEN,[B-]12(OC3C(C(OC3(O1)CO)CO)O)OC4C(C(OC4(O2)CO)...,calcium fructoborate,calcium fructoborate results in increased expr...,increases^expression,774.258,367.0900,367.104785,-5.2466,2.630,...,4,4,0,0,0,0,0,0,0,0
2,EIF4EBP1,[B-]12(OC3C(C(OC3(O1)CO)CO)O)OC4C(C(OC4(O2)CO)...,calcium fructoborate,calcium fructoborate affects the expression of...,affects^expression,774.258,367.0900,367.104785,-5.2466,2.630,...,4,4,0,0,0,0,0,0,0,0
3,TSC2,[B-]12(OC3C(C(OC3(O1)CO)CO)O)OC4C(C(OC4(O2)CO)...,calcium fructoborate,calcium fructoborate results in increased expr...,increases^expression,774.258,367.0900,367.104785,-5.2466,2.630,...,4,4,0,0,0,0,0,0,0,0
4,TP53,[Be+2].[O-]S(=O)(=O)[O-],beryllium sulfate,beryllium sulfate promotes the reaction [TP53 ...,affects^binding|increases^reaction,105.074,98.0779,97.967380,-3.4103,1.524,...,0,0,0,0,0,0,0,0,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54396,RELA,S1[As]2S[As]3[As]1S[As]2S3,tetraarsenic tetrasulfide,tetraarsenic tetrasulfide results in decreased...,decreases^expression,427.952,427.9520,427.574656,0.0000,-0.530,...,6,6,0,0,0,0,0,0,0,0
54397,RELA,S1[As]2S[As]3[As]1S[As]2S3,tetraarsenic tetrasulfide,tetraarsenic tetrasulfide results in decreased...,decreases^expression,427.952,427.9520,427.574656,0.0000,-0.530,...,6,6,0,0,0,0,0,0,0,0
54398,MTOR,S1[As]2S[As]3[As]1S[As]2S3,tetraarsenic tetrasulfide,tetraarsenic tetrasulfide promotes the reactio...,decreases^expression|increases^reaction,427.952,427.9520,427.574656,0.0000,-0.530,...,6,6,0,0,0,0,0,0,0,0
54399,EIF4EBP1,S1[As]2S[As]3[As]1S[As]2S3,tetraarsenic tetrasulfide,tetraarsenic tetrasulfide promotes the reactio...,decreases^expression|increases^reaction,427.952,427.9520,427.574656,0.0000,-0.530,...,6,6,0,0,0,0,0,0,0,0


In [ ]:
# Filtrar las filas de ctd_data que SÍ están en LeadLikeness
coincidencias_ctd = ctd_data[ctd_data['Name'].isin(LeadLikeness['Name'])]

In [ ]:
# Filtrar las filas de ctd_data que SÍ están en LeadLikeness y eliminar el resto
coincidencias_ctd = coincidencias_ctd.drop_duplicates(subset='Name')

In [ ]:
coincidencias_ctd

,GeneSymbol,SMILES,Name,Interaction,InteractionActions,Total Molweight,Molweight,Monoisotopic Mass,cLogP,cLogS,...,Saturated Hetero-Rings,Non-Aromatic Hetero-Rings,Hetero-Aromatic Rings,Amides,Amines,Alkyl-Amines,Aromatic Amines,Aromatic Nitrogens,Basic Nitrogens,Acidic Oxygens
6189,AKT1,C=C1CC2C(C(C3C1CC(C3=C)O)OC(=O)C(=C)CO)C(=C)C(...,hemistepsin,2-(4-morpholinyl)-8-phenyl-4H-1-benzopyran-4-o...,decreases^phosphorylation|decreases^reaction,346.378,346.378,346.141640,1.3523,-2.710,...,1,1,0,0,0,0,0,0,0,0
6259,BMI1,C=CC(=O)N1CCCC(C1)N2C3=NC=NC(=C3C(=N2)C4=CC=C(...,ibrutinib,ibrutinib results in decreased expression of B...,decreases^expression,440.506,440.506,440.196074,3.6932,-6.321,...,1,1,2,1,1,0,1,4,0,0
6427,JUN,C=CC12CC(C3C(C1C(=C)C(=O)OC2)OC(=O)C3=C)OC(=O)...,vernodalin,MAPK8 protein affects the reaction [vernodalin...,affects^reaction|increases^phosphorylation,360.361,360.361,360.120905,0.8315,-2.525,...,2,2,0,0,0,0,0,0,0,0
7814,MAPK3,C1=C(C2=C(C(=O)C(=O)C3=C2NC(=C3)C(=O)O)N=C1C(=...,PQQ Cofactor,[Rotenone co-treated with PQQ Cofactor] result...,affects^cotreatment|increases^phosphorylation,330.208,330.208,330.012418,-0.8801,-2.648,...,0,0,2,0,0,0,0,2,0,3
8213,TP53,C1=C2C3=C(C(=C1O)O)OC(=O)C4=CC(=C(C(=C43)OC2=O...,Ellagic Acid,Ellagic Acid promotes the reaction [Quercetin ...,increases^phosphorylation|increases^reaction,302.194,302.194,302.006270,1.2774,-3.286,...,0,2,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50475,MAPK14,COC1=CC=CC(=C1C2=CC(=O)C3=C(C(=C(C(=C3O2)OC)OC...,skullcapflavone II,skullcapflavone II binds to MAPK14 protein,affects^binding,374.344,374.344,374.100170,2.4014,-3.224,...,0,1,0,0,0,0,0,0,0,0
51055,CCND1,COC1=CC=CC2=C1NC(=C2)C3=C4C(=NC=NN4C(=N3)C5CCC...,OSI 027,[apabetalone co-treated with OSI 027] results ...,affects^cotreatment|decreases^expression,406.445,406.445,406.175339,1.4893,-4.270,...,0,0,3,0,1,0,1,5,1,1
51188,CDK2,COC1=CN=C(C2=C1C3=C(N2)C(=CC=C3)O)C=C,picrasidine I,picrasidine I affects the expression of CDK2 p...,affects^expression,240.261,240.261,240.089878,2.3562,-3.398,...,0,0,2,0,0,0,0,2,0,0
51749,TP53,CS(=O)C1=CC=C(C=C1)C2=NC(=C(N2)C3=CC=NC=C3)C4=...,SB 203580,SB 203580 inhibits the reaction [[Cisplatin co...,affects^cotreatment|decreases^reaction|increas...,377.442,377.442,377.099810,3.5687,-5.242,...,0,0,2,0,0,0,0,3,1,0


In [ ]:
# Guardar coincidencias_ctd en un archivo CSV
coincidencias_ctd.to_csv('LeadLikeness_DesMol.csv', index=False)
